# 27 — Routage : distance routée et distance excessive

Lit les **demandes de routage** écrites par le notebook 22 (`data/routing_requests.parquet`), jusqu'à deux par leg : **OD brute** (point d'entrée interpolé → dernier waypoint) et **OD corrigée** (1re → dernière position recalée par le map matching). Les envoie au service local **xyt_routing**, et écrit `legs/routing_by_leg.parquet` : colonnes `routing_*` pour l'OD brute, `routing_matched_od_*` pour l'OD corrigée. **Relancer ensuite le notebook 22** : il lit cette table et calcule la distance excessive (observée − routée).

## Avant de lancer : démarrer le service (dans un terminal)

1. **Docker Desktop** lancé (icône de la baleine dans la barre de menus, état *running*).
2. Conteneurs OSRM, depuis le dossier `xyt_routing` :
   ```bash
   docker compose --profile road-ch-extended-200km up -d osrm-car-ch-extended-200km osrm-car-distance-ch-extended-200km
   docker compose ps          # les deux conteneurs doivent être « running »
   ```
3. API Python (laisser ce terminal ouvert) :
   ```bash
   XYT_ROUTING_SERVICE_CONFIG=configs/service.ch_extended_200km.yaml .venv/bin/xyt-routing-api
   ```

La cellule « Vérification du service » envoie une route test et explique ce qui ne va pas si le service ne répond pas.

In [1]:
CONFIG_NAME = "full_waypoints.yaml"
OUTPUT_DIR = "outputs/pl23"

BASE_URL = "http://127.0.0.1:8010"
MODE = "car_ch_extended_200km"     # alias du graphe dans la config du service
WEIGHTING = "distance"             # plus court chemin
BATCH_SIZE = 500
MAX_ROUTES = None                  # ex. 200 pour un premier essai ; None = toutes les demandes
FORCE_REROUTE = False              # True : ignorer le cache et tout rerouter
INCLUDE_GEOMETRY = True            # géométries routées pour la carte de comparaison
COMPARISON_MAP_LEGS = 30

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from parking_search_prediction.config import load_config
from parking_search_prediction.routing_xyt import check_service, load_requests, run_routing, write_comparison_map

config = load_config(PROJECT_ROOT / "config" / CONFIG_NAME)
config["paths"]["output_dir"] = OUTPUT_DIR
config.setdefault("routing", {}).update({
    "base_url": BASE_URL, "mode": MODE, "weighting": WEIGHTING, "batch_size": BATCH_SIZE,
    "max_routes": MAX_ROUTES, "force_reroute": FORCE_REROUTE, "include_geometry": INCLUDE_GEOMETRY,
})

## Vérification du service

In [3]:
pd.Series(check_service(config), name="service de routage")

url                                http://127.0.0.1:8010/routes
mode                                      car_ch_extended_200km
weighting                                              distance
test_route_distance_m                                    1654.3
network_id                  ch-extended-200km-osrm-car-distance
network_version          osm-ch_extended_200km-1418d8a09b76af1b
engine                                                     osrm
Name: service de routage, dtype: object

## Demandes à router (écrites par le notebook 22)

In [4]:
requests = load_requests(config)
display(pd.Series({"demandes": len(requests), "legs": requests["leg_id"].nunique(),
                   "type de route": requests["route_type"].unique().tolist(),
                   "écrites le": requests["created_at"].iloc[0] if len(requests) else None}, name="demandes"))
requests.head()

demandes                                                     63458
legs                                                         35896
type de route    [buffer_entry_to_leg_end, matched_entry_to_mat...
écrites le                               2026-11-11T10:54:45+00:00
Name: demandes, dtype: object

,leg_id,route_type,origin_lon,origin_lat,destination_lon,destination_lat,observed_distance_m,observed_duration_s,routing_request_id,buffer_radius_m,created_at
0,00181b52-062f-4fa4-a0c3-f23241cca9c4,buffer_entry_to_leg_end,6.213930,46.392351,6.223176,46.386027,1507.937908,154.416806,6817849d26e4278d,1000.0,2026-11-11T10:54:45+00:00
1,001a0cf0-bb47-4133-94aa-816f837d85a9,buffer_entry_to_leg_end,6.374263,46.071838,6.387174,46.071431,1249.309219,172.842572,6ef444b2b457e4b6,1000.0,2026-11-11T10:54:45+00:00
2,001f4cd8-f023-4138-aba9-51df64d580d5,buffer_entry_to_leg_end,6.362297,46.270967,6.371160,46.264399,1105.413989,151.459095,c083500ee4d3861c,1000.0,2026-11-11T10:54:45+00:00
3,0021e19d-89d2-43c1-8e6d-f92bc78b1000,buffer_entry_to_leg_end,6.120750,46.181697,6.112430,46.188591,1814.470076,216.928767,066f89e2ef75bfc7,1000.0,2026-11-11T10:54:45+00:00
4,0034e051-e3c3-4e3c-9709-e63e676a9044,buffer_entry_to_leg_end,6.313826,46.269574,6.323429,46.275621,1285.915148,136.969220,2d77b349628a7bf2,1000.0,2026-11-11T10:54:45+00:00


## Routage

Les routes déjà présentes dans le cache (`data/routing_cache.parquet`) ne sont pas renvoyées ; les erreurs ne sont pas mises en cache et seront retentées au prochain run.

In [5]:
routing = run_routing(config)
display(pd.Series({k: v for k, v in routing["summary"].items() if not isinstance(v, (dict, list))}, name="routage"))
display(pd.DataFrame({"OD brute": routing["summary"]["status_counts"],
                      "OD corrigée": routing["summary"]["status_counts_matched_od"]}).fillna(0).astype(int))
pd.DataFrame({"OD brute (m)": routing["summary"]["dist_excess_m"],
              "OD corrigée (m)": routing["summary"]["dist_excess_matched_od_m"]})

[routage] 63/63 routes — 1 s


requests                                                        63458
routed_this_run                                                    63
from_cache                                                      63395
ok                                                              63458
errors                                                              0
not_yet_routed                                                      0
collection_table    /Users/maximiliantrique/Documents/Projet_pytho...
seconds                                                           6.3
Name: routage, dtype: object

,OD brute,OD corrigée
not_requested,623304,631638
routed,35896,27562


,OD brute (m),OD corrigée (m)
count,35896.0,27562.0
mean,-143.9,72.0
std,1372.4,255.1
min,-46123.0,-1063.5
25%,-43.1,0.0
50%,-4.3,0.1
75%,30.9,0.3
90%,191.8,184.0
max,6416.6,5381.8


## Contrôle visuel : trace GPS (orange) et les deux plus courts chemins — OD brute (bleu) et OD corrigée (violet)

In [6]:
map_path = write_comparison_map(config, routing["by_leg"], n_legs=COMPARISON_MAP_LEGS)
display(HTML(f"<a href='{Path(map_path).as_uri()}' target='_blank'>Ouvrir la carte de comparaison</a>") if map_path
        else "Pas de géométrie routée à afficher.")

## Suite

Relancer le notebook **22** : la distance excessive (`spatial_dist_excess_m`) et le critère `detect_dist_excess_min_m` sont alors actifs, et la mesure du cruising passe sur le détour (`search_measure = excess_if_available`). Le notebook 41 en tiendra compte automatiquement.